# M3A — Análisis de experimentos (A1, A2, A3, B)


In [ ]:
import json, sqlite3
import numpy as np
from pathlib import Path
from scipy import stats

REPO = Path.cwd().parents[2]  # experiments/M3A/open_source -> repo root
INDEX = REPO / 'experiments/M3A/m3a_index_2000.json'
DBS = {
    'qwen2b': REPO / 'experiments/M3A/open_source/results/qwen3vl_2b/qwen3vl_cache.db',
    'qwen8b': REPO / 'experiments/M3A/open_source/results/qwen3vl_8b/qwen3vl_cache.db',
    'wave':   REPO / 'experiments/M3A/open_source/results/WAVE7B/wave_cache.db',
    'ge2':    REPO / 'experiments/M3A/google_embeddings2/results/m3a_ge2.db',
}
index = json.load(open(INDEX, encoding='utf-8'))
by_id = {e['raw_id']: e for e in index}
print('index entries:', len(index))

In [ ]:
def load_embeddings(db_path, modality):
    """Return {raw_id: np.float32 vector} for a modality (empty if DB missing)."""
    if not Path(db_path).exists():
        return {}
    c = sqlite3.connect(str(db_path))
    rows = c.execute('SELECT raw_id, vector FROM embeddings WHERE modality=?', (modality,)).fetchall()
    c.close()
    return {rid: np.frombuffer(v, dtype=np.float32) for rid, v in rows}

def l2(v):
    n = np.linalg.norm(v)
    return v / n if n > 1e-9 else v

def cos(a, b):
    return float(np.dot(l2(a), l2(b)))

def summarize(real, fake, name):
    """Paired real-vs-fake summary: mean diff, t-test, paper-style accuracy."""
    real, fake = np.asarray(real), np.asarray(fake)
    diff = real - fake
    acc = float(np.mean(real > fake))  # real pair scores higher -> correct
    t, p = stats.ttest_rel(real, fake) if len(diff) > 1 else (np.nan, np.nan)
    return {'exp': name, 'n': len(diff), 'mean_real': round(real.mean(), 4),
            'mean_fake': round(fake.mean(), 4), 'mean_diff': round(diff.mean(), 4),
            'acc': round(acc, 4), 't': round(float(t), 2), 'p': f'{p:.1e}'}

## A1 / A2 — OOC vídeo↔texto y vídeo↔transcript (MM re-pairing)


In [ ]:
def ooc_experiment(V, X, mm_key, rng):
    """V, X: dicts raw_id->vec. mm_key in {'text','audio','video'} selects the swap.
    Returns (real, fake_dataset, fake_random) lists of cosines over common ids."""
    ids = [r for r in V if r in X]
    real, fake_ds, fake_rnd = [], [], []
    pool = ids[:]
    for rid in ids:
        rc = cos(V[rid], X[rid])
        # dataset mismatch (restricted to in-subset source)
        src = by_id[rid]['mm_sources'].get(mm_key)
        if mm_key == 'video':
            fk = cos(V[src], X[rid]) if src in V else None
        else:
            fk = cos(V[rid], X[src]) if src in X else None
        # random control
        j = rng.choice(pool)
        while j == rid:
            j = rng.choice(pool)
        fr = cos(V[rid], X[j]) if mm_key != 'video' else cos(V[j], X[rid])
        if fk is not None:
            real.append(rc); fake_ds.append(fk); fake_rnd.append(fr)
    return real, fake_ds, fake_rnd

rng = np.random.default_rng(0)
results = []
for model, db in DBS.items():
    V  = load_embeddings(db, 'video')
    T  = load_embeddings(db, 'text_summary')
    Tr = load_embeddings(db, 'transcript')
    if not V:
        print(f'[skip] {model}: no embeddings yet'); continue
    # A1: video<->summary, swap text
    r, fds, frn = ooc_experiment(V, T, 'text', rng)
    if r: results.append({'model': model, **summarize(r, fds, 'A1 V-T (MM text, dataset)')})
    if r: results.append({'model': model, **summarize(r, frn, 'A1 V-T (random control)')})
    # A2: video<->transcript, swap audio
    r, fds, frn = ooc_experiment(V, Tr, 'audio', rng)
    if r: results.append({'model': model, **summarize(r, fds, 'A2 V-Tr (MM audio, dataset)')})
    if r: results.append({'model': model, **summarize(r, frn, 'A2 V-Tr (random control)')})

import pandas as pd
pd.DataFrame(results)

## A3 — Triángulo de coherencia (V · Summary · Transcript)


In [ ]:
def perimeter(v, t, tr):
    return (1 - cos(v, t)) + (1 - cos(v, tr)) + (1 - cos(t, tr))

rows = []
for model, db in DBS.items():
    V  = load_embeddings(db, 'video')
    T  = load_embeddings(db, 'text_summary')
    Tr = load_embeddings(db, 'transcript')
    ids = [r for r in V if r in T and r in Tr]
    if not ids:
        continue
    real, fake = [], []
    for rid in ids:
        src = by_id[rid]['mm_sources'].get('text')
        if src not in T:
            continue
        real.append(perimeter(V[rid], T[rid], Tr[rid]))
        fake.append(perimeter(V[rid], T[src], Tr[rid]))
    if real:
        # higher perimeter on fake -> correct: invert sign for paper-style acc
        rows.append({'model': model, **summarize(fake, real, 'A3 perimeter (fake>real)')})
pd.DataFrame(rows)

## B — NEM (fact-check por subtipo)


In [ ]:
NEM_SUBS = ['person', 'location', 'organization', 'complete']
rows = []
for model, db in DBS.items():
    V = load_embeddings(db, 'video')
    T = load_embeddings(db, 'text_summary')
    if not V:
        continue
    for sub in NEM_SUBS:
        Tn = load_embeddings(db, f'text_nem_{sub}')
        ids = [r for r in V if r in T and r in Tn]
        if not ids:
            continue
        real = [cos(V[r], T[r]) for r in ids]
        fake = [cos(V[r], Tn[r]) for r in ids]
        rows.append({'model': model, **summarize(real, fake, f'B NEM-{sub}')})
pd.DataFrame(rows)

## D (opcional) — Estratificación / OOD
